# Código para estraer mensajes de meta

In [6]:
import requests
import pandas as pd

# --- PASO A: PEGA AQUÍ TU TOKEN NUEVO ---
token = 'EAAQ9FrwwOUABQtmr9O17uro3UWXX3bZAGWRiTyXIZB4KVm2mexZAb2HANYjd9qG20RCYZAfWk8mHuoSUXymsZBlY5PnxNabegXMEAei2uSeoCDCyu32UZACZBSQpZBHtPJNehVEZAIr0DAXpZBcFFVDQBhMM9wX256jswhMuAeTPKGNczUiEcpw4xqNJgVdOofbNZCkhCiUBQIuLdjYRuDOJUveZCk7xhADwV3lcfZBt0CSwZD' 

# --- PASO B: ID DE LA POCION (YA ESTÁ LISTO) ---
page_id = '895431323874532'

def obtener_todo_el_contenido():
    # Agregamos '?limit=25' para ser explícitos, aunque es el defecto.
    url = f"https://graph.facebook.com/v24.0/{page_id}/published_posts?fields=id,message,created_time&limit=25&access_token={token}"
    
    try:
        res = requests.get(url).json()
        if 'error' in res:
            print("Error obteniendo publicaciones:", res['error']['message'])
            return []
        return res.get('data', [])
    except Exception as e:
        print(f"Error de conexión: {e}")
        return []

def obtener_comentarios(object_id):
    # Pedimos hasta 100 comentarios por publicación para no perdernos nada
    url = f"https://graph.facebook.com/v24.0/{object_id}/comments?fields=from,message,created_time&limit=100&access_token={token}"
    res = requests.get(url).json()
    return res.get('data', [])

# --- EJECUCIÓN ---
print("1. Descargando las últimas 25 publicaciones...")
contenidos = obtener_todo_el_contenido()

lista_final = []

if contenidos:
    print(f"   -> Se encontraron {len(contenidos)} publicaciones. Extrayendo comentarios...")
    
    for i, item in enumerate(contenidos):
        post_id = item['id']
        # Usamos los primeros 30 caracteres del mensaje para identificarlo visualmente
        mensaje_corto = item.get('message', 'Sin título')[:30].replace('\n', ' ')
        
        print(f"   [{i+1}/25] Procesando: {mensaje_corto}...")
        
        comentarios = obtener_comentarios(post_id)
        for c in comentarios:
            lista_final.append({
                'Post_ID': post_id,
                'Fecha_Post': item.get('created_time'), # Agregué fecha del post para referencia
                'Usuario': c.get('from', {}).get('name', 'Anónimo'),
                'Comentario': c.get('message'),
                'Fecha_Comentario': c.get('created_time')
            })

    # Guardar en Excel
    df = pd.DataFrame(lista_final)
    nombre_archivo = 'comentarios_ultimos_25_posts.xlsx'
    df.to_excel(nombre_archivo, index=False)
    print(f"\n✅ ¡LISTO! Se guardaron {len(lista_final)} comentarios en '{nombre_archivo}'")

else:
    print("❌ No se pudieron descargar las publicaciones. Revisa el Token.")

1. Descargando las últimas 25 publicaciones...
   -> Se encontraron 25 publicaciones. Extrayendo comentarios...
   [1/25] Procesando: Hay cosas por las que no tenem...
   [2/25] Procesando: Tres SPOILER en televisión nac...
   [3/25] Procesando: El olor a plancha en el cabell...
   [4/25] Procesando: 2 de Enero para algunos, día l...
   [5/25] Procesando: Este 2025 no habría podido ser...
   [6/25] Procesando: Más que bloopers, es Pocionlan...
   [7/25] Procesando: Hay cosas que quiero contarles...
   [8/25] Procesando: Si de regalos hablamos me atre...
   [9/25] Procesando: Siempre podemos ser una mejor ...
   [10/25] Procesando: Sin título...
   [11/25] Procesando: Confirmado, La PROMO no vuelve...
   [12/25] Procesando: ¡Qué no te pase! ☝🏼 No te qued...
   [13/25] Procesando: Fue un año sin Promo 😩 sabemos...
   [14/25] Procesando: ⸻  PROMO POCIÓN 25% OFF a part...
   [15/25] Procesando: Sin título...
   [16/25] Procesando: Mañana PROMO POCIÓN - La Salva...
   [17/25] Procesando: A 

In [ ]:
# me/accounts?fields=name,access_token,tasks   # Obtener los tokens de las paginas

In [1]:
import requests
import pandas as pd
import json

# Tus variables
# page_id = '61585396901295' 
# post_id = '122100993531179896'
# pocion = 100064772436300 ,  895431323874532




page_id = '895431323874532' 
post_id = '899348319324577' 
access_token = 'EAAQ9FrwwOUABQdWinGZCvfDqvknse4Kk8TJRWbH94C7AlZAuARFB0toGPh7tKeK17BNhtRJK23CULZC6ioBIYG4HE1LeacXiXMMhdAYYx2nIpTLGQBx2ZBpoHl1HdIbwiBhxwS65qimNubLRgYQJAZCVZABg45kLaZBPdIIZB6ikDC7tEgn2kr0KLZAkguZCAssnZAlhhfT5ZBKe81ZARadxGLqDNP0u4wfU5EioxBPCrl0UZD' 

# RECOMENDACIÓN: Agrega 'fields' a la URL para asegurar que pides los datos explícitamente
url = f'https://graph.facebook.com/v24.0/{page_id}_{post_id}/comments?access_token={access_token}&fields=from,message,created_time'

response = requests.request("GET", url)
data = json.loads(response.text)

# Función corregida con manejo de errores (safe access)
def get_comment(comment):
    # Usa .get() para evitar el KeyError si el campo falta
    user_info = comment.get('from', {})
    user_name = user_info.get('name', 'Usuario Desconocido')
    
    return {
        'name': user_name,
        'time': comment.get('created_time'),
        'message': comment.get('message')
    }

# Procesar los datos
if 'data' in data:
    excel_data = list(map(get_comment, data['data']))
    df = pd.DataFrame(excel_data)
    df.to_excel(r"C:\Users\Dataa\Desktop\comentarios.xlsx", index=False)
    print("Archivo 'comments.xlsx' creado exitosamente.")
else:
    print("Error en la respuesta de la API:", data)

Error en la respuesta de la API: {'error': {'message': 'Error validating access token: Session has expired on Wednesday, 24-Dec-25 12:00:00 PST. The current time is Tuesday, 20-Jan-26 12:37:27 PST.', 'type': 'OAuthException', 'code': 190, 'error_subcode': 463, 'fbtrace_id': 'AjrE7rLMOvsFbntrCJ9M-Zp'}}


2

In [ ]:
import requests
import pandas as pd

# Configuración
token = 'EAAQ9FrwwOUABQdWinGZCvfDqvknse4Kk8TJRWbH94C7AlZAuARFB0toGPh7tKeK17BNhtRJK23CULZC6ioBIYG4HE1LeacXiXMMhdAYYx2nIpTLGQBx2ZBpoHl1HdIbwiBhxwS65qimNubLRgYQJAZCVZABg45kLaZBPdIIZB6ikDC7tEgn2kr0KLZAkguZCAssnZAlhhfT5ZBKe81ZARadxGLqDNP0u4wfU5EioxBPCrl0UZD' 
page_id = '895431323874532'

token = '' 
page_id = ''
def obtener_todo_el_contenido():
    # Esta consulta trae publicaciones del muro Y videos/reels
    url = f"https://graph.facebook.com/v24.0/{page_id}/published_posts?fields=id,message,display_subtext,attachments&access_token={token}"
    res = requests.get(url).json()
    return res.get('data', [])

def obtener_comentarios(object_id):
    url = f"https://graph.facebook.com/v24.0/{object_id}/comments?fields=from,message,created_time&access_token={token}"
    res = requests.get(url).json()
    return res.get('data', [])

# Ejemplo de ejecución
contenidos = obtener_todo_el_contenido()
lista_final = []

for item in contenidos:
    post_id = item['id']
    print(f"Procesando contenido ID: {post_id}")
    
    comentarios = obtener_comentarios(post_id)
    for c in comentarios:
        lista_final.append({
            'Post_ID': post_id,
            'Usuario': c.get('from', {}).get('name', 'Anónimo'),
            'Comentario': c.get('message'),
            'Fecha': c.get('created_time')
        })

df = pd.DataFrame(lista_final)
df.to_excel('comentarios_multicontenido.xlsx', index=False)

# Todos los comentarios

In [5]:
import requests
import pandas as pd

# Configuración
token = 'EAAQ9FrwwOUABQoYSnE4VpLHAlo0x4Kx46iV4pmZCPPxygkUVV291gFRbngfO7hcvBLINzVs5PxGbs9nsPXwQ8YW0BZBhrVlxK4kEnCtmHZC488Bycl6hO8niE19ys4La5mw4zt4ZBKOrbeTbkO1mGbzEPafTfzjMvxOlsW3ketYj4jJPwPgDCvTlTeT2Yfd0VFZCYFtEaHgZCn8pYdCFTGrUewnaXZC1Jx3WkTqAwnORRwZD' 
page_id = '895431323874532'

def descargar_todo(post_id, token):
    # 1. Aumentamos el límite por petición a 100 (el máximo permitido)
    # 2. Usamos filter=stream para que el orden sea cronológico
    url = f"https://graph.facebook.com/v24.0/{post_id}/comments?limit=100&filter=stream&fields=from,message,created_time&access_token={token}"
    
    comentarios_totales = []
    
    while url:
        response = requests.get(url)
        data = response.json()
        
        if 'data' in data:
            for c in data['data']:
                user_info = c.get('from', {})
                comentarios_totales.append({
                    'Nombre': user_info.get('name', 'Anónimo'),
                    'Mensaje': c.get('message'),
                    'Fecha': c.get('created_time')
                })
        
        # Aquí está el truco: buscamos si hay una página siguiente
        url = data.get('paging', {}).get('next')
        
        if url:
            print(f"Descargados {len(comentarios_totales)} comentarios... buscando más...")

    return comentarios_totales

# Ejecución
resultados = descargar_todo(post_id, access_token)
df = pd.DataFrame(resultados)
df.to_excel('todos_los_comentarios.xlsx', index=False)
print(f"✅ Proceso finalizado. Total: {len(resultados)} comentarios guardados.")

✅ Proceso finalizado. Total: 0 comentarios guardados.


# Descarga completa comentarios de facebook

In [39]:
import requests
import pandas as pd
import time

# --- CONFIGURACIÓN ---
ACCESS_TOKEN = "EAAQ9FrwwOUABQe2e1E4EdJAOpsupJdlD3y32xbmPMFYubNVl9ItPy4rzXAfQqfz3VBIMEsFO1UaLdClGmwluxkPDZAVdb13P2XdqlN5sj5lp7mhZBYOwN5jgllneuZAIbU2zz0sdq4Pcia8zblIGpw7iitoa38Xdw6ZCZARLK4xY6ilYHjmxLSZCnGvQ3D1SBda1wEh0xNBNl9OtUppt1M8qobj9MFImQXm0CExJQZD"

# ACCESS_TOKEN = 'EAAQ9FrwwOUABQdWinGZCvfDqvknse4Kk8TJRWbH94C7AlZAuARFB0toGPh7tKeK17BNhtRJK23CULZC6ioBIYG4HE1LeacXiXMMhdAYYx2nIpTLGQBx2ZBpoHl1HdIbwiBhxwS65qimNubLRgYQJAZCVZABg45kLaZBPdIIZB6ikDC7tEgn2kr0KLZAkguZCAssnZAlhhfT5ZBKe81ZARadxGLqDNP0u4wfU5EioxBPCrl0UZD'
PAGE_ID = '895431323874532'

def obtener_todos_los_videos(token):
    print("Buscando lista de videos...")
    videos = []
    # Endpoint para videos subidos
    url = f"https://graph.facebook.com/v24.0/{PAGE_ID}/videos?fields=id,description,created_time&limit=50&access_token={token}"
    
    while url:
        res = requests.get(url).json()
        if 'data' in res:
            videos.extend(res['data'])
        url = res.get('paging', {}).get('next')
        print(f"Videos encontrados hasta ahora: {len(videos)}")
    return videos

def obtener_comentarios_de_video(video_id, token):
    comentarios = []
    url = f"https://graph.facebook.com/v24.0/{video_id}/comments?fields=from,message,created_time&limit=100&access_token={token}"
    
    while url:
        res = requests.get(url).json()
        if 'data' in res:
            for c in res['data']:
                comentarios.append({
                    'Video_ID': video_id,
                    'Fecha_Comentario': c.get('created_time'),
                    'Usuario': c.get('from', {}).get('name', 'Anónimo'),
                    'Mensaje': c.get('message')
                })
        url = res.get('paging', {}).get('next')
    return comentarios

# --- EJECUCIÓN PRINCIPAL ---
todos_los_videos = obtener_todos_los_videos(ACCESS_TOKEN)
base_datos_comentarios = []

print(f"\nIniciando descarga de comentarios de {len(todos_los_videos)} videos...")

for i, video in enumerate(todos_los_videos):
    v_id = video['id']
    desc = video.get('description', 'Sin descripción')[:50] # Tomamos los primeros 50 caracteres
    
    print(f"[{i+1}/{len(todos_los_videos)}] Procesando video: {v_id} ({desc}...)")
    
    comentarios_video = obtener_comentarios_de_video(v_id, ACCESS_TOKEN)
    base_datos_comentarios.extend(comentarios_video)
    
    # Pequeña pausa para no saturar la API (Rate Limit)
    time.sleep(0.5)

# Guardar todo en un Excel
if base_datos_comentarios:
    df = pd.DataFrame(base_datos_comentarios)
    df.to_excel('HISTORICO_COMENTARIOS_VIDEOS.xlsx', index=False)
    print(f"\n✅ ¡TERMINADO! Se guardaron {len(base_datos_comentarios)} comentarios en 'HISTORICO_COMENTARIOS_VIDEOS.xlsx'")
else:
    print("\n❌ No se encontraron comentarios.")

Buscando lista de videos...
Videos encontrados hasta ahora: 50
Videos encontrados hasta ahora: 100
Videos encontrados hasta ahora: 150
Videos encontrados hasta ahora: 200
Videos encontrados hasta ahora: 250
Videos encontrados hasta ahora: 300
Videos encontrados hasta ahora: 350
Videos encontrados hasta ahora: 400
Videos encontrados hasta ahora: 450
Videos encontrados hasta ahora: 450

Iniciando descarga de comentarios de 450 videos...
[1/450] Procesando video: 886733737377298 (Sin descripción...)
[2/450] Procesando video: 2052563648854674 (Sin descripción...)
[3/450] Procesando video: 853307564227557 (Sin descripción...)
[4/450] Procesando video: 1383437699827976 (Sin descripción...)
[5/450] Procesando video: 755596133554556 (Sin descripción...)
[6/450] Procesando video: 4523894687934345 (Sin descripción...)
[7/450] Procesando video: 889357560095713 (Sin descripción...)
[8/450] Procesando video: 2172520846606677 (Sin descripción...)
[9/450] Procesando video: 2955589531303329 (Sin descr

# Instagran Completo

In [8]:
import requests
import pandas as pd
import time

# --- TUS DATOS ---
ACCESS_TOKEN = 'EAAQ9FrwwOUABQiVgUJzxNVfpEfoW22ukZBGn21izTqT7oZBFUcoJu0k584WDIQPS2XuAqZAZByWyE2rb5kpq5ZBc2JreZCNNHV1Q1cC4pFlZBRQ2Ov6nySEiOXtrVQZAIhkAcjVAPh4DoZBmVDnSectPM42AC157AV82Ou9vS2t9hk7U0MWeYrpRrTxcW20QeksjR2Cqb7uImw4jh2CZC9k3oPTPsUNGxlxy9nRm9D2cmsGd9PtupaZBjIwtbEDRxskbvaImjfNzakaMRJ7VPDK8eZCQR5j0'
# Asegúrate de usar el IG User ID (no el de Facebook)
IG_USER_ID = '17841402208128553' 

def descargar_instagram_completo():
    # 1. Solicitamos 50 posts por lote para ir más rápido (limit=50)
    url_media = f"https://graph.facebook.com/v24.0/{IG_USER_ID}/media?fields=id,caption,media_product_type,timestamp&limit=50&access_token={ACCESS_TOKEN}"
    
    todos_los_datos = []
    contador_posts = 0
    
    print("🚀 INICIANDO DESCARGA DEL HISTORIAL COMPLETO...")

    # --- BUCLE EXTERNO: Recorre las páginas de PUBLICACIONES ---
    while url_media:
        try:
            response = requests.get(url_media)
            data_media = response.json()
            
            if 'error' in data_media:
                print(f"❌ Error en API: {data_media['error']['message']}")
                break

            posts_batch = data_media.get('data', [])
            
            if not posts_batch:
                print("No hay más publicaciones.")
                break

            # Procesamos cada post del lote actual
            for post in posts_batch:
                contador_posts += 1
                media_id = post['id']
                caption = post.get('caption', 'Sin descripción')[:40].replace('\n', ' ')
                tipo = post.get('media_product_type')
                
                print(f"[{contador_posts}] Analizando: {caption}...")
                
                # --- BUCLE INTERNO: Recorre los COMENTARIOS de ese post ---
                url_comments = f"https://graph.facebook.com/v24.0/{media_id}/comments?fields=text,username,timestamp&limit=100&access_token={ACCESS_TOKEN}"
                
                while url_comments:
                    res_comm = requests.get(url_comments).json()
                    comments_data = res_comm.get('data', [])
                    
                    for c in comments_data:
                        todos_los_datos.append({
                            'ID_Post': media_id,
                            'Fecha_Post': post.get('timestamp'),
                            'Tipo_Post': tipo,
                            'Descripción_Post': post.get('caption'),
                            'Usuario_Comentario': c.get('username'),
                            'Comentario': c.get('text'),
                            'Fecha_Comentario': c.get('timestamp')
                        })
                    
                    # Paginación de COMENTARIOS (Siguiente página de comentarios)
                    url_comments = res_comm.get('paging', {}).get('next')

            # --- Paginación de PUBLICACIONES (Siguiente lote de 50 posts) ---
            url_media = data_media.get('paging', {}).get('next')
            
            if url_media:
                print("⏳ Cargando posts más antiguos... (Espera unos segundos)")
                time.sleep(1) # Pausa pequeña para cuidar el límite de la API
                
        except Exception as e:
            print(f"⚠️ Ocurrió un error inesperado: {e}")
            break

    return todos_los_datos

# --- EJECUCIÓN ---
datos_finales = descargar_instagram_completo()

if datos_finales:
    print(f"\n✅ PROCESO FINALIZADO.")
    print(f"📊 Se extrajeron {len(datos_finales)} comentarios en total.")
    
    # Guardar en Excel
    df = pd.DataFrame(datos_finales)
    df.to_excel('Instagram_Historial_Completo.xlsx', index=False)
    print("💾 Archivo guardado como: Instagram_Historial_Completo.xlsx")
else:
    print("❌ No se encontraron datos o hubo un error.")

🚀 INICIANDO DESCARGA DEL HISTORIAL COMPLETO...
[1] Analizando: Hay cosas por las que no tenemos que suf...
[2] Analizando: Tres SPOILER en televisión nacional por ...
[3] Analizando: El olor a plancha en el cabello si exist...


KeyboardInterrupt: 

In [9]:
import requests
import pandas as pd
import time

# --- TUS DATOS ---
# --- TUS DATOS ---
ACCESS_TOKEN = 'EAAQ9FrwwOUABQiVgUJzxNVfpEfoW22ukZBGn21izTqT7oZBFUcoJu0k584WDIQPS2XuAqZAZByWyE2rb5kpq5ZBc2JreZCNNHV1Q1cC4pFlZBRQ2Ov6nySEiOXtrVQZAIhkAcjVAPh4DoZBmVDnSectPM42AC157AV82Ou9vS2t9hk7U0MWeYrpRrTxcW20QeksjR2Cqb7uImw4jh2CZC9k3oPTPsUNGxlxy9nRm9D2cmsGd9PtupaZBjIwtbEDRxskbvaImjfNzakaMRJ7VPDK8eZCQR5j0'
# Asegúrate de usar el IG User ID (no el de Facebook)
IG_USER_ID = '17841402208128553' 

def obtener_comentarios_videos_ig():
    # 1. Pedimos 'media_type' para distinguir videos de fotos
    # Pedimos limit=100 para tener un buen lote inicial y buscar los videos ahí
    url_media = f"https://graph.facebook.com/v24.0/{IG_USER_ID}/media?fields=id,caption,media_type,timestamp&limit=100&access_token={ACCESS_TOKEN}"
    
    data_final = []
    contador_videos = 0
    LIMITE_VIDEOS = 25 # Queremos exactamente 25 videos
    
    print(f"🚀 Buscando los últimos {LIMITE_VIDEOS} videos...")

    while url_media and contador_videos < LIMITE_VIDEOS:
        try:
            res_media = requests.get(url_media).json()
            items = res_media.get('data', [])
            
            if not items:
                break # No hay más publicaciones

            for item in items:
                # --- FILTRO: Solo procesamos si es VIDEO ---
                if item.get('media_type') == 'VIDEO':
                    contador_videos += 1
                    
                    media_id = item['id']
                    # Usamos los primeros 50 caracteres del caption como "Nombre del Video"
                    nombre_video = item.get('caption', 'Video sin descripción')[:50].replace('\n', ' ')
                    
                    print(f"[{contador_videos}/{LIMITE_VIDEOS}] Extrayendo comentarios de: {nombre_video}...")

                    # 2. Obtener comentarios de este video
                    url_comm = f"https://graph.facebook.com/v24.0/{media_id}/comments?fields=text,username,timestamp&limit=100&access_token={ACCESS_TOKEN}"
                    
                    while url_comm:
                        res_comm = requests.get(url_comm).json()
                        comentarios = res_comm.get('data', [])
                        
                        for c in comentarios:
                            data_final.append({
                                'ID_Video': media_id,
                                'Nombre_Video': item.get('caption'), # El "Nombre" completo
                                'Fecha_Video': item.get('timestamp'),
                                'Usuario_Comentario': c.get('username'), # Persona que comentó
                                'Comentario': c.get('text'),
                                'Fecha_Comentario': c.get('timestamp')
                            })
                        
                        # Paginación de comentarios
                        url_comm = res_comm.get('paging', {}).get('next')

                    # Si ya llegamos a 25 videos, rompemos el bucle for
                    if contador_videos >= LIMITE_VIDEOS:
                        break
            
            # Paginación de publicaciones (Siguiente lote si no hemos llegado a 25 videos)
            url_media = res_media.get('paging', {}).get('next')

        except Exception as e:
            print(f"Error: {e}")
            break

    return data_final

# Ejecución
print("Iniciando proceso...")
comentarios = obtener_comentarios_videos_ig()

if comentarios:
    df = pd.DataFrame(comentarios)
    # Reordenamos columnas para que se vea ordenado en Excel
    columnas = ['ID_Video', 'Nombre_Video', 'Fecha_Video', 'Usuario_Comentario', 'Comentario', 'Fecha_Comentario']
    df = df[columnas]
    
    df.to_excel('comentarios_ultimos_25_videos.xlsx', index=False)
    print(f"✅ ¡Listo! Se guardaron {len(comentarios)} comentarios de los últimos 25 videos.")
else:
    print("No se encontraron comentarios o hubo un error con el token.")

Iniciando proceso...
🚀 Buscando los últimos 25 videos...
[1/25] Extrayendo comentarios de: Hay cosas por las que no tenemos que sufrir y tene...
[2/25] Extrayendo comentarios de: Tres SPOILER en televisión nacional por nuestra pr...
[3/25] Extrayendo comentarios de: El olor a plancha en el cabello si existe muchacha...
[4/25] Extrayendo comentarios de: 2 de Enero para algunos, día laboral innecesario p...
[5/25] Extrayendo comentarios de: Este 2025 no habría podido ser mejor, fue perfecto...
[6/25] Extrayendo comentarios de: Más que bloopers, es Pocionland.  Aquí un poquito ...
[7/25] Extrayendo comentarios de: Hay cosas que quiero contarles ya mis muchachas, e...
[8/25] Extrayendo comentarios de: Siempre podemos ser una mejor versión y ese fue el...
[9/25] Extrayendo comentarios de: Confirmado, La PROMO no vuelve hasta dentro de un ...
[10/25] Extrayendo comentarios de: ¡Qué no te pase! ☝🏼 No te quedes esperando un año ...
 Por... Extrayendo comentarios de: Fue un año sin Promo 😩 sabe